# Rinascente Milano store - data preparation

## Introduction
The dataset contains transactional and customer data from Rinascente, an Italian luxury department store chain, capturing purchases, discounts, and customer demographics. It provides insights into shopping behavior, loyalty program participation, and sales performance across different product categories.

Here, the dataset is cleaned and prepared to use.

---------

A mechanism is first of all defined to ensure that all necessary Python libraries required to run this notebook are installed.

In [1]:
# Imports system utilities and subprocess for managing installations
import sys
import subprocess

# Defines the list of libraries required for the application
required_libraries = ["pandas", "numpy", "openpyxl"]

# Iterates through each required library
for lib in required_libraries:
    try:
        # Tries to import the library to check if it is already installed
        __import__(lib)
    except ImportError:
        # If the library is missing, prints a message and installs it using pip
        print(f"Installing {lib}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", lib])

Import of libraries used

In [2]:
import sys
import subprocess
import pandas as pd
import numpy as np
import openpyxl
import re

### Dataset overview

In [3]:
df_store_milano = pd.read_excel("STORE MILANO.xlsx") # Load dataset
display(df_store_milano.head()) # Display first rows

,Customer Account ID,Actual Tier ID,Customer Creation Date DESC,GENERATIONS,Age ID,Gender DESC,Region ID,District ID,Town ID,Marketing Consent Flag ID,...,Call Consent ID,Is Ecommerce Client ID,Sales Date DESC,Alternative Division DESC_COMPLETE,Actual Ownership Type DESC,Brand DESC_COMPLETE,Customer Sales (val),Customer Discount (%),Customer Discount (val),Customer Sales (qty)
0,55,runneR,09Jun2020,GEN Y,44,FEMALE,LOMBARDIA,MI,VAPRIO D'ADDA,1,...,0,1,31Oct2023,901-COSMETICS,Diretto,2FT-SENSAI,253.00,0.283286,100.0,3
1,70,staRter,06Jun2020,GEN X,51,FEMALE,LOMBARDIA,MI,NaN,1,...,1,1,31Oct2023,205-UNDERWEAR & BEACHWEAR,Diretto,430-POLO RALPH LAUREN,49.95,0.000000,0.0,1
2,261,loveR,23Jun2020,GEN X,52,FEMALE,LOMBARDIA,MI,NaN,0,...,0,1,31Oct2023,801-HOME,Diretto,1VL-HAY,17.00,0.000000,0.0,1
3,261,loveR,23Jun2020,GEN X,52,FEMALE,LOMBARDIA,MI,NaN,0,...,0,1,31Oct2023,801-HOME,Diretto,953-DECO DESIGN,95.00,0.000000,0.0,1
4,329,heRo,22Jun2020,BABY BOOMER,68,FEMALE,LOMBARDIA,MI,MILANO,1,...,1,0,31Oct2023,101-WOMEN RTW,Diretto,1QZ-DES PHEMMES,545.00,0.268456,200.0,1


In [4]:
print(f'Dataset dimension: {df_store_milano.shape}')

Dataset dimension: (439103, 24)


In [5]:
print(f'Dataset info:\n')
print(df_store_milano.info())

Dataset info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439103 entries, 0 to 439102
Data columns (total 24 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   Customer Account ID                 439103 non-null  int64  
 1   Actual Tier ID                      438407 non-null  object 
 2   Customer Creation Date DESC         439103 non-null  object 
 3   GENERATIONS                         439103 non-null  object 
 4   Age ID                              439103 non-null  int64  
 5   Gender DESC                         439103 non-null  object 
 6   Region ID                           391573 non-null  object 
 7   District ID                         388155 non-null  object 
 8   Town ID                             378566 non-null  object 
 9   Marketing Consent Flag ID           439103 non-null  int64  
 10  Primary eMail Present ID            439103 non-null  bool   
 11  Email Conse

### Data type and format transformation and handling null values

Data type and format transformation

In [7]:
# date columns to datetime format
date_columns = ['Customer Creation Date DESC', 'Sales Date DESC']

for col in date_columns:
    df_store_milano[col] = pd.to_datetime(df_store_milano[col], format='%d%b%Y', errors='coerce')

In [8]:
# check the range date of date columns
for col in date_columns:
    min_date = df_store_milano[col].min()
    max_date = df_store_milano[col].max()
    print(f"Date range in '{col}': {min_date} -- {max_date}")

Date range in 'Customer Creation Date DESC': 2008-04-25 00:00:00 -- 2024-08-25 00:00:00
Date range in 'Sales Date DESC': 2023-08-28 00:00:00 -- 2024-08-25 00:00:00


In [9]:
# categorical columns
categorical_columns = ['Actual Tier ID', 'GENERATIONS', 'Gender DESC', 'Region ID', 'District ID', 'Town ID', 'Alternative Division DESC_COMPLETE', 'Actual Ownership Type DESC', 'Brand DESC_COMPLETE']
df_store_milano[categorical_columns] = df_store_milano[categorical_columns].astype('category')

In [10]:
# binary columns to boolean
bool_columns = ['Marketing Consent Flag ID', 'Email Consent ID', 'SMS Consent ID', 'Call Consent ID', 'Is Ecommerce Client ID']
df_store_milano[bool_columns] = df_store_milano[bool_columns].astype(bool)

In [12]:
# Check again data types
print(df_store_milano.dtypes)

Customer Account ID                            int64
Actual Tier ID                              category
Customer Creation Date DESC           datetime64[ns]
GENERATIONS                                 category
Age ID                                         int64
Gender DESC                                 category
Region ID                                   category
District ID                                 category
Town ID                                     category
Marketing Consent Flag ID                       bool
Primary eMail Present ID                        bool
Email Consent ID                                bool
Mobile Phone Present ID                         bool
SMS Consent ID                                  bool
Call Consent ID                                 bool
Is Ecommerce Client ID                          bool
Sales Date DESC                       datetime64[ns]
Alternative Division DESC_COMPLETE          category
Actual Ownership Type DESC                  ca

In [13]:
# upper case specific columns
uppercase_columns = ["GENERATIONS", "Gender DESC", "Region ID", "District ID", "Town ID"]

df_store_milano[uppercase_columns] = df_store_milano[uppercase_columns].apply(lambda x: x.astype(str).str.strip().str.upper())

Check unique values in specific columns for possible further transformation

In [14]:
# first are checked columns with few unique values
columns_to_check = ['Actual Tier ID', 'GENERATIONS', 'Gender DESC', 'Alternative Division DESC_COMPLETE', 'Actual Ownership Type DESC']

# print unique values sorted alphabetically
for col in columns_to_check:
    unique_values = sorted(df_store_milano[col].dropna().unique()) # Na values are ignored for the moment
    print(f"Unique values in '{col}':")
    print(unique_values, "\n")

Unique values in 'Actual Tier ID':
['heRo', 'loveR', 'runneR', 'staRter'] 

Unique values in 'GENERATIONS':
['BABY BOOMER', 'GEN X', 'GEN Y', 'GEN Z'] 

Unique values in 'Gender DESC':
['FEMALE', 'MALE', 'NOT SPECIFIED'] 

Unique values in 'Alternative Division DESC_COMPLETE':
['101-WOMEN RTW', '102-WOMEN ACC.', '201-MEN', '205-UNDERWEAR & BEACHWEAR', '301-CHILDREN', '601-FOOD & BEVERAGE', '701-OFFICE & TRAVEL', '801-HOME', '901-COSMETICS', '911-VARIE'] 

Unique values in 'Actual Ownership Type DESC':
['Concession', 'Diretto'] 



The same is done with columns with a lot of unique values

In [15]:
# first are checked columns with few unique values
columns_to_check = ['Region ID', 'District ID', 'Town ID']

# print unique values sorted alphabetically
for col in columns_to_check:
    unique_values = sorted(df_store_milano[col].dropna().unique()) # Na values are ignored for the moment
    print(f"Unique values in '{col} ({len(unique_values)}):")
    print(unique_values, "\n")

Unique values in 'Region ID (21):
['ABRUZZO', 'BASILICATA', 'CALABRIA', 'CAMPANIA', 'EMILIA-ROMAGNA', 'FRIULI-VENEZIA GIULIA', 'LAZIO', 'LIGURIA', 'LOMBARDIA', 'MARCHE', 'MOLISE', 'NAN', 'PIEMONTE', 'PUGLIA', 'SARDEGNA', 'SICILIA', 'TOSCANA', 'TRENTINO-ALTO ADIGE', 'UMBRIA', "VALLE D'AOSTA", 'VENETO'] 

Unique values in 'District ID (110):
['AG', 'AL', 'AN', 'AO', 'AP', 'AQ', 'AR', 'AT', 'AV', 'BA', 'BG', 'BI', 'BL', 'BN', 'BO', 'BR', 'BS', 'BT', 'BZ', 'CA', 'CB', 'CE', 'CH', 'CI', 'CL', 'CN', 'CO', 'CR', 'CS', 'CT', 'CZ', 'EN', 'FC', 'FE', 'FG', 'FI', 'FM', 'FR', 'GE', 'GO', 'GR', 'IM', 'IS', 'KR', 'LC', 'LE', 'LI', 'LO', 'LT', 'LU', 'MB', 'MC', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NAN', 'NO', 'NU', 'OG', 'OR', 'OT', 'PA', 'PC', 'PD', 'PE', 'PG', 'PI', 'PN', 'PO', 'PR', 'PT', 'PU', 'PV', 'PZ', 'RA', 'RC', 'RE', 'RG', 'RI', 'RM', 'RN', 'RO', 'SA', 'SI', 'SO', 'SP', 'SR', 'SS', 'SV', 'TA', 'TE', 'TN', 'TO', 'TP', 'TR', 'TS', 'TV', 'UD', 'VA', 'VB', 'VC', 'VE', 'VI', 'VR', 'VS', 'VT', 'V

Town ID column is cleaned from unwanted and unexpected values.

In [16]:
unwanted_values = ['.', '..', '00047', '00136', '1', '10050', '20093', '20122', '20132', '20141', '22010', '22063', '31100', '37042', '37121', '52100']

# Replace unwanted values with NaN
df_store_milano['Town ID'] = df_store_milano['Town ID'].replace(unwanted_values, np.nan)

# Function to remove leading numbers and keep only the town name (ex. 20153 MILANO -> MILANO)
def clean_town_name(town):
    if isinstance(town, str):  # Ensure the value is a string
        return re.sub(r'^\d+\s*', '', town)  # Remove leading numbers and spaces
    return town  # Return as is if not a string

# Apply the cleaning function
df_store_milano['Town ID'] = df_store_milano['Town ID'].apply(clean_town_name)

# Check cleaned data
print(df_store_milano['Town ID'].unique())  # Display unique values after cleaning

["VAPRIO D'ADDA" 'NAN' 'MILANO' ... 'SAN BASSANO' 'SAN LAZZARO' 'MONASTIR']


NAN strings are subsituted with actual null values in columns "Region ID", "District ID" and "Town ID"

In [17]:
df_store_milano[columns_to_check] = df_store_milano[columns_to_check].replace('NAN', np.nan)

Handling missing values

In [18]:
print(df_store_milano.isnull().sum())

Customer Account ID                       0
Actual Tier ID                          696
Customer Creation Date DESC               0
GENERATIONS                               0
Age ID                                    0
Gender DESC                               0
Region ID                             47530
District ID                           50948
Town ID                               60633
Marketing Consent Flag ID                 0
Primary eMail Present ID                  0
Email Consent ID                          0
Mobile Phone Present ID                   0
SMS Consent ID                            0
Call Consent ID                           0
Is Ecommerce Client ID                    0
Sales Date DESC                           0
Alternative Division DESC_COMPLETE        0
Actual Ownership Type DESC                0
Brand DESC_COMPLETE                       0
Customer Sales (val)                      0
Customer Discount (%)                  4230
Customer Discount (val)         

Rows with null values in "Actual Tier ID", "Region ID", "District ID" or "Town ID" are dropped.

In [19]:
col_with_null_values = ["Actual Tier ID", "Region ID", "District ID", "Town ID"]
for col in col_with_null_values:
    df_store_milano = df_store_milano.dropna(subset=[col])

Before populating Customer Discount (%), ranges of values are checked accross the four columns **Customer Sales (val)**, **Customer Discount (%)**, **Customer Discount (val)** and **Customer Sales (qty)**

In [20]:
columns_to_check = ['Customer Sales (val)', 'Customer Discount (%)', 'Customer Discount (val)', 'Customer Sales (qty)']

for col in columns_to_check:
    print(f"Range of values in '{col}':")
    print(f"Min: {df_store_milano[col].min()}")
    print(f"Max: {df_store_milano[col].max()}\n") 

Range of values in 'Customer Sales (val)':
Min: -10600.0
Max: 36900.0

Range of values in 'Customer Discount (%)':
Min: -119.4
Max: 8.38

Range of values in 'Customer Discount (val)':
Min: -1549.99
Max: 10482.92

Range of values in 'Customer Sales (qty)':
Min: -25
Max: 446



In [21]:
# count negative values in each colum
negative_counts = {}

for column in columns_to_check:
    negative_count = (df_store_milano[column] < 0).sum()
    negative_counts[column] = negative_count

for column, count in negative_counts.items():
    print(f"Column '{column}' contains {count} negative values.")

Column 'Customer Sales (val)' contains 7288 negative values.
Column 'Customer Discount (%)' contains 282 negative values.
Column 'Customer Discount (val)' contains 4538 negative values.
Column 'Customer Sales (qty)' contains 6890 negative values.


To check if a negative sign '-' was set by accident on some of those variables, the absolute values of `Customer Sales (val)` and ``Customer Discount (val)`` are stored in new columns and used to calculate also a new column of ``Customer Discount (%)`` to compare with the absolute values of the original ``Customer Discount (%)``.

In [22]:
# Create absolute value versions of Customer Sales (val) and Customer Discount (val) columns
df_store_milano['Abs Customer Sales (val)'] = df_store_milano['Customer Sales (val)'].abs()
df_store_milano['Abs Customer Discount (val)'] = df_store_milano['Customer Discount (val)'].abs()


# Calculate "Recomputed Customer Discount (%)" using the formula
df_store_milano['Recomputed Customer Discount (%)'] = (
    df_store_milano['Abs Customer Discount (val)'] / (df_store_milano['Abs Customer Sales (val)'] + df_store_milano['Abs Customer Discount (val)'])
)

# Create "Abs Customer Discount (%)" for direct comparison
df_store_milano['Abs Customer Discount (%)'] = df_store_milano['Customer Discount (%)'].abs()

# Identify mismatches (ignoring nulls)
mismatches = df_store_milano[
    (~df_store_milano['Customer Discount (%)'].isna()) & 
    (df_store_milano['Recomputed Customer Discount (%)'].round(5) != df_store_milano['Abs Customer Discount (%)'].round(5))
]

# Display number of mismatches
print(f"Number of mismatches found: {len(mismatches)}")

Number of mismatches found: 242


Mismatch are found. So, those rows are eliminated. Original columns with negative values are dropped. New columns are named with the original column names.

In [23]:
df_store_milano = df_store_milano.drop(mismatches.index) # drop mismatches

# Drop original columns with negative values
df_store_milano.drop(columns=['Customer Sales (val)', 'Customer Discount (val)',
                              'Customer Discount (%)', 'Recomputed Customer Discount (%)'
                              ], inplace=True)

#Rename absolute value columns back to original names
df_store_milano.rename(columns={'Abs Customer Sales (val)': 'Customer Sales (val)',
                                'Abs Customer Discount (val)': 'Customer Discount (val)',
                                'Abs Customer Discount (%)': 'Customer Discount (%)'
                                }, inplace=True)

Now, null values in Customer Discount (%) can be re-populated using the same formula ``Customer Discount (val)`` / ``(Customer Sales (val) + Customer Discount (val))``. Obviously, if there is a 0 in one of the two columns used in the formula, it means that the customer have not purchased anything and that the Customer Discount (%) must be 0 as well.

In [25]:
mask_na = df_store_milano['Customer Discount (%)'].isna()  # Identify rows with NaN
mask_zero = (df_store_milano['Customer Sales (val)'] == 0) | (df_store_milano['Customer Discount (val)'] == 0)  # Identify where either value is 0

# First, fill NAN values with 0 where either column is 0
df_store_milano.loc[mask_na & mask_zero, 'Customer Discount (%)'] = 0

# Then, apply the formula to the remaining null values
df_store_milano.loc[mask_na & ~mask_zero, 'Customer Discount (%)'] = (
    df_store_milano['Customer Discount (val)'] / (df_store_milano['Customer Sales (val)'] + df_store_milano['Customer Discount (val)'])
    )

Now, the following rows are also dropped:

- Rows with negative values in Customer Sales (qty) - there is no way to do a check over those and eventually keep them, as has been done with other sale columns
- Rows where Customer Sales (qty) is >0 but Customer Sales (val) =0 and viceversa

In [26]:
df_store_milano = df_store_milano[df_store_milano['Customer Sales (qty)'] >= 0] # drop rows with negative values

condition_1 = (df_store_milano['Customer Sales (qty)'] > 0) & (df_store_milano['Customer Sales (val)'] == 0) # Identify rows where sales quantity is positive but sales value is 0
condition_2 = (df_store_milano['Customer Sales (val)'] > 0) & (df_store_milano['Customer Sales (qty)'] == 0) # Identify rows where sales value is positive but sales quantity is 0

rows_to_drop = df_store_milano[condition_1 | condition_2] # Combine both conditions

print(f"Total rows to be dropped: {len(rows_to_drop)}")

Total rows to be dropped: 934


In [27]:
# Drop the identified rows
df_store_milano = df_store_milano.drop(rows_to_drop.index)

In [28]:
# make sure negative values are not present anymore
negative_counts = {}
for column in columns_to_check:
    negative_count = (df_store_milano[column] < 0).sum()
    negative_counts[column] = negative_count

for column, count in negative_counts.items():
    print(f"Column '{column}' contains {count} negative values.")

#print again null values in each column and final shape of the dataframe
print(f'\nFinal check over null values:\n{df_store_milano.isnull().sum()}\n')
print(f'Final Dataframe shape: {df_store_milano.shape}')

Column 'Customer Sales (val)' contains 0 negative values.
Column 'Customer Discount (%)' contains 0 negative values.
Column 'Customer Discount (val)' contains 0 negative values.
Column 'Customer Sales (qty)' contains 0 negative values.

Final check over null values:
Customer Account ID                   0
Actual Tier ID                        0
Customer Creation Date DESC           0
GENERATIONS                           0
Age ID                                0
Gender DESC                           0
Region ID                             0
District ID                           0
Town ID                               0
Marketing Consent Flag ID             0
Primary eMail Present ID              0
Email Consent ID                      0
Mobile Phone Present ID               0
SMS Consent ID                        0
Call Consent ID                       0
Is Ecommerce Client ID                0
Sales Date DESC                       0
Alternative Division DESC_COMPLETE    0
Actual Owners

In [27]:
df_store_milano.to_excel("STORE MILANO CLEANED.xlsx", index=False, engine="openpyxl")

In [28]:
df_store_milano.to_csv("STORE MILANO CLEANED.csv", index=False, sep=";")